In [1]:
import os
print(os.getcwd())  # Veja onde você está

d:\Projetos_2026_ML\Self-Healing Governado\notebooks


In [2]:
# Defina ou importe antes de execute_rule
from dataclasses import dataclass

@dataclass
class RuleResult:
    # campos conforme o restante do notebook
    pass

def execute_rule(rule, bundle, context) -> RuleResult:
    ...

In [3]:
import os
import inspect

# Descobrir a estrutura
project_root = r"D:\Projetos_2026\ML\Self-Healing Governado"

for root, dirs, files in os.walk(project_root):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
                if 'class RuleResult' in content or 'RuleOutput' in content:
                    print(f"Encontrado em: {filepath}")
                    break

In [4]:
from time import perf_counter
from typing import Any, Mapping




VALID_STATUSES = {"PASS", "FAIL", "ERROR"}
VALID_SEVERITIES = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}


def _duration_ms(start: float) -> float:
    return round((perf_counter() - start) * 1000, 3)


def _coerce_count(value: Any) -> int:
    if value is None:
        return 0

    count = int(value)

    if count < 0:
        raise ValueError(f"count must be non-negative, got {count}")

    return count


def _normalize_status(value: Any, count: int) -> str:
    if value is None:
        return "PASS" if count == 0 else "FAIL"

    status = str(value).upper()

    if status not in VALID_STATUSES:
        raise ValueError(f"invalid rule status: {value!r}")

    return status


def _normalize_severity(value: Any) -> str:
    if value is None:
        return "MEDIUM"

    severity = str(value).upper()

    if severity not in VALID_SEVERITIES:
        raise ValueError(f"invalid rule severity: {value!r}")

    return severity


def execute_rule(rule, bundle, context) -> RuleResult:
    start = perf_counter()

    rule_name = getattr(rule, "name", rule.__class__.__name__)
    rule_id = getattr(rule, "rule_id", rule_name)

    try:
        raw: RuleOutput = rule.run(bundle, context) or {}

        if not isinstance(raw, Mapping):
            raise TypeError(
                f"rule.run() must return a mapping-compatible RuleOutput, got {type(raw).__name__}"
            )

        count = _coerce_count(raw.get("count"))
        status = _normalize_status(raw.get("status"), count)
        severity = _normalize_severity(raw.get("severity"))

        return RuleResult(
            rule_id=rule_id,
            rule_name=rule_name,
            status=status,
            severity=severity,
            count=count,
            message=raw.get("message"),
            details=raw.get("details") or [],
            duration_ms=_duration_ms(start),
        )

    except Exception as exc:
        return RuleResult(
            rule_id=rule_id,
            rule_name=rule_name,
            status="ERROR",
            severity="CRITICAL",
            count=0,
            message=f"{type(exc).__name__}: {exc}",
            details=[],
            duration_ms=_duration_ms(start),
        )